# FINPLE Step 108-13A — US Price Metrics Overlay Colab

This notebook generates US price metrics overlay chunks for FINPLE.

Metrics generated:

```text
expectedCagr
priceCagr10y
mdd
beta
dataYears
metricsStatus
```

Policy: `expectedCagr` uses close-price CAGR, not total-return CAGR. Dividend yield is handled separately.


In [ ]:
# 1. Install dependencies
!pip -q install yfinance pandas


## 2. Upload required files

Upload these three files from the repository:

```text
finple_app_candidates_6000_balanced_v1.csv
build_us_price_metrics_overlay_chunked.py
combine_us_price_metrics_chunks.py
```


In [ ]:
from google.colab import files as colab_files
uploaded = colab_files.upload()


## 3. First test run — 20 rows


In [ ]:
!python build_us_price_metrics_overlay_chunked.py \
  --input finple_app_candidates_6000_balanced_v1.csv \
  --out-runtime us_price_metrics_overlay_20260528_part0000_0020.csv \
  --out-audit us_price_metrics_overlay_20260528_part0000_0020_audit.csv \
  --out-summary us_price_metrics_overlay_20260528_part0000_0020_summary.json \
  --as-of 2026-05-28 \
  --start 0 \
  --limit 20 \
  --checkpoint-every 5


## 4. Inspect 20-row test output


In [ ]:
import pandas as pd, json, os
runtime_file = 'us_price_metrics_overlay_20260528_part0000_0020.csv'
summary_file = 'us_price_metrics_overlay_20260528_part0000_0020_summary.json'
print('runtime exists:', os.path.exists(runtime_file), runtime_file)
print('summary exists:', os.path.exists(summary_file), summary_file)
test_df = pd.read_csv(runtime_file)
display(test_df.head(30))
with open(summary_file, 'r', encoding='utf-8') as f:
    print(json.dumps(json.load(f), ensure_ascii=False, indent=2))


## 5. Production chunk run — 100 rows

After the test succeeds, run chunks of 100. Change `--start` and file names for each chunk.


In [ ]:
!python build_us_price_metrics_overlay_chunked.py \
  --input finple_app_candidates_6000_balanced_v1.csv \
  --out-runtime us_price_metrics_overlay_20260528_part0000_0100.csv \
  --out-audit us_price_metrics_overlay_20260528_part0000_0100_audit.csv \
  --out-summary us_price_metrics_overlay_20260528_part0000_0100_summary.json \
  --as-of 2026-05-28 \
  --start 0 \
  --limit 100 \
  --checkpoint-every 25


## 6. Combine chunks


In [ ]:
!python combine_us_price_metrics_chunks.py \
  --pattern 'us_price_metrics_overlay_20260528_part*.csv' \
  --out-runtime us_price_metrics_overlay_20260528.csv \
  --out-summary us_price_metrics_overlay_20260528_summary.json


## 7. Inspect combined sample


In [ ]:
df = pd.read_csv('us_price_metrics_overlay_20260528.csv')
sample = ['SPY','VOO','QQQ','SCHD','JEPI','JEPQ','AAPL','MSFT','KO','JNJ','JPM','XOM','AMZN','TSLA','SNOW','AAT','AGNC']
display(df[df['ticker'].isin(sample)].sort_values('ticker'))
display(df.head(20))
with open('us_price_metrics_overlay_20260528_summary.json', 'r', encoding='utf-8') as f:
    print(json.dumps(json.load(f), ensure_ascii=False, indent=2))


## 8. Download outputs


In [ ]:
from google.colab import files as colab_files
for target in [
    'us_price_metrics_overlay_20260528.csv',
    'us_price_metrics_overlay_20260528_summary.json',
]:
    if os.path.exists(target):
        colab_files.download(target)
    else:
        print('missing:', target)
